# Ground Truth Cleaning

Karena data ground_truth.csv memiliki beberapa ketidakkonsistenan format, dan karena data ground truth akans sering digunakan, akan lebih baik kalau kita menggunakan data ground truth yang sudah konsisten formatnya. 

In [1]:
import csv
import re
import pandas as pd
from pathlib import Path

RAW_PATH = Path("ground_truth.csv")          # adjust path if needed
OUTPUT_PATH = Path("fixed_ground_truth.csv")


In [2]:
with open(RAW_PATH, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

# drop the header and any stray blank lines (trailing newline at EOF, etc.)
header = raw_lines[0]
raw_rows = [l for l in raw_lines[1:] if l.strip() != ""]

print(f"Header: {header}")
print(f"Data rows found: {len(raw_rows)}")


Header: filename,name,birth_date,address
Data rows found: 632


In [3]:
def parse_row(line: str) -> list[str]:
    if line.startswith('"') and line.endswith('"'):
        inner = line[1:-1].replace('""', '"')
        fields = next(csv.reader([inner]))
    else:
        fields = next(csv.reader([line]))
    return fields


In [4]:
parsed_records = []
parse_failures = []

for i, line in enumerate(raw_rows, start=1):
    fields = parse_row(line)
    if len(fields) != 4:
        parse_failures.append((i, len(fields), line[:120]))
    else:
        parsed_records.append(fields)

print(f"Successfully parsed: {len(parsed_records)} / {len(raw_rows)}")
print(f"Parse failures: {len(parse_failures)}")

if parse_failures:
    print("\nFailed rows (line_no, n_fields, preview):")
    for fail in parse_failures:
        print(fail)


Successfully parsed: 632 / 632
Parse failures: 0


In [5]:
df = pd.DataFrame(parsed_records, columns=["filename", "name", "birth_date", "address"])
df["filename"] = df["filename"].str.strip()
df["name"] = df["name"].str.strip()
df["birth_date"] = df["birth_date"].str.strip()
df["address"] = df["address"].str.strip()

print(df.shape)
df.head(10)


(632, 4)


,filename,name,birth_date,address
0,image_001.jpg,HAMZAH BIN KAMMAPU,1965-02-11,"PT 1160 P, JALAN KENANGA, 20400 KUALA TERENGGA..."
1,image_002.jpg,RAZALI BIN AHMAD,1959-02-24,"167, KAMPUNG BANGGOL AIR LILEH, BATU ENAM, 212..."
2,image_003.jpg,NOR ATHIRAH NAJWA BINTI RAZALI,2000-12-30,"167, KAMPUNG BANGGOL AIR LILEH, BATU 6, 21200 ..."
3,image_004.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
4,image_005.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
5,image_006.jpg,MUHAMAD ZAMRI BIN SHAFEE,1986-06-13,MARKAS TENTERA DARAT CAWANGAN SUMBER MANUSIA K...
6,image_007.jpg,MOHAMAD FADZALIISAM BIN ITHNIN,1979-07-19,"K-G-16 JALAN 2/6, TAMAN SETAPAK INDAH, 53300 K..."
7,image_008.jpg,MOHAMAD ADAM HARRIS BIN MOHAMAD FADZALIISAM,2016-04-07,"BLOK E5-1-1, TAMAN MELATI, SETAPAK, 53100 KUAL..."
8,image_009.jpg,AFFANDY BIN OTHMAN,1981-09-19,"NO 1592, JALAN SIRAM, 12100 BUTTERWORTH, PULAU..."
9,image_010.jpg,FATIN NUR NAJWA BINTI FAMY,2000-08-12,"NO 29 JALAN PLATINUM 7/50, SEKSYEN 7, 40000 SH..."


In [6]:
audit_log = {}

# duplicate filenames
audit_log["duplicate_filenames"] = int(df["filename"].duplicated().sum())

# empty fields
audit_log["empty_name"] = int((df["name"] == "").sum())
audit_log["empty_birth_date"] = int((df["birth_date"] == "").sum())
audit_log["empty_address"] = int((df["address"] == "").sum())

# birth_date not in strict YYYY-MM-DD format (e.g. year-only values like "1983")
date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
non_standard_dates = df[~df["birth_date"].str.match(date_pattern)]
audit_log["non_standard_birth_date_rows"] = int(len(non_standard_dates))

print("Audit summary:")
for k, v in audit_log.items():
    print(f"  {k}: {v}")

if len(non_standard_dates) > 0:
    print("\nSample non-standard birth_date rows:")
    print(non_standard_dates[["filename", "name", "birth_date"]].head(10).to_string(index=False))


Audit summary:
  duplicate_filenames: 0
  empty_name: 0
  empty_birth_date: 0
  empty_address: 520
  non_standard_birth_date_rows: 20

Sample non-standard birth_date rows:
     filename             name birth_date
image_273.jpg ABDELLAH SLIMANI       1983
image_274.jpg ABDELLAH SLIMANI       1983
image_275.jpg ABDELLAH SLIMANI       1983
image_276.jpg ABDELLAH SLIMANI       1983
image_277.jpg ABDELLAH SLIMANI       1983
image_278.jpg ABDELLAH SLIMANI       1983
image_279.jpg ABDELLAH SLIMANI       1983
image_280.jpg ABDELLAH SLIMANI       1983
image_281.jpg ABDELLAH SLIMANI       1983
image_282.jpg ABDELLAH SLIMANI       1983


In [7]:
df.to_csv(OUTPUT_PATH, index=False, quoting=csv.QUOTE_MINIMAL, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")


Saved: E:\comp\compfest_final_proj\fixed_ground_truth.csv


In [8]:
check_df = pd.read_csv(OUTPUT_PATH)
print(check_df.shape)
assert check_df.shape == df.shape, "Row/column count mismatch after reload!"
assert list(check_df.columns) == ["filename", "name", "birth_date", "address"], "Column mismatch!"
print("Reload check passed — fixed_ground_truth.csv is standard-parseable.")
check_df.head(10)


(632, 4)
Reload check passed — fixed_ground_truth.csv is standard-parseable.


,filename,name,birth_date,address
0,image_001.jpg,HAMZAH BIN KAMMAPU,1965-02-11,"PT 1160 P, JALAN KENANGA, 20400 KUALA TERENGGA..."
1,image_002.jpg,RAZALI BIN AHMAD,1959-02-24,"167, KAMPUNG BANGGOL AIR LILEH, BATU ENAM, 212..."
2,image_003.jpg,NOR ATHIRAH NAJWA BINTI RAZALI,2000-12-30,"167, KAMPUNG BANGGOL AIR LILEH, BATU 6, 21200 ..."
3,image_004.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
4,image_005.jpg,SAFURA BINTI ABDUL KHALIM,1982-07-27,"106 JALAN PERTIWI, TAMAN MALURI, 55100 KUALA L..."
5,image_006.jpg,MUHAMAD ZAMRI BIN SHAFEE,1986-06-13,MARKAS TENTERA DARAT CAWANGAN SUMBER MANUSIA K...
6,image_007.jpg,MOHAMAD FADZALIISAM BIN ITHNIN,1979-07-19,"K-G-16 JALAN 2/6, TAMAN SETAPAK INDAH, 53300 K..."
7,image_008.jpg,MOHAMAD ADAM HARRIS BIN MOHAMAD FADZALIISAM,2016-04-07,"BLOK E5-1-1, TAMAN MELATI, SETAPAK, 53100 KUAL..."
8,image_009.jpg,AFFANDY BIN OTHMAN,1981-09-19,"NO 1592, JALAN SIRAM, 12100 BUTTERWORTH, PULAU..."
9,image_010.jpg,FATIN NUR NAJWA BINTI FAMY,2000-08-12,"NO 29 JALAN PLATINUM 7/50, SEKSYEN 7, 40000 SH..."
